# Analisis Kemampuan Metakognitif (Pre–Post)

Notebook ini mengolah data angket metakognitif (4 dimensi, skor 4–16 per dimensi, total 16–64).
Metodologi sesuai Bab III: N-Gain Hake, paired t-test, independent t-test (Welch), Cohen's d.

**Cakupan:**
1. Validasi data (rentang, konsistensi total, duplikasi)
2. Statistik deskriptif pre dan post per kelompok dan dimensi
3. Kategorisasi skor metakognitif
4. Gain absolut dan N-Gain ternormalisasi (Hake, 1998)
5. Uji normalitas Jarque–Bera pada gain
6. Paired sample t-test (perubahan dalam kelompok)
7. Independent sample t-test / Welch (perbedaan N-Gain antarkelompok)
8. Effect size Cohen's d / Hedges' g
9. Analisis per dimensi
10. Korelasi metakognitif postes × keterampilan berbicara postes


In [1]:
import csv, math
from pathlib import Path
from collections import Counter
from statistics import mean, median, stdev, variance

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'field_test' / 'metakognitif.csv').exists():
            return p
    raise FileNotFoundError("Root proyek tidak ditemukan")

ROOT = find_root()
META_PATH = ROOT / 'data' / 'field_test' / 'metakognitif.csv'
POST_PATH = ROOT / 'data' / 'field_test' / 'keterampilan_berbicara_postes.csv'

DIMS = ['planning', 'monitoring', 'evaluation', 'integratif']
DIM_LABEL = {'planning':'Perencanaan','monitoring':'Pemantauan',
             'evaluation':'Evaluasi','integratif':'Integratif (Feynman)'}
MAX_TOTAL = 64.0
MAX_DIM   = 16.0

def load_meta():
    with open(META_PATH, encoding='utf-8-sig', newline='') as f:
        raw = list(csv.DictReader(f))
    result = []
    for row in raw:
        rec = {'id': row['id'], 'kelompok': row['kelompok'], 'nama': row['nama']}
        for phase in ('pre', 'post'):
            for dim in DIMS:
                col = f'{phase}_{dim}'
                rec[col] = float(row[col]) if row.get(col) else None
            col_t = f'{phase}_total'
            rec[col_t] = float(row[col_t]) if row.get(col_t) else None
        result.append(rec)
    return result

def load_post_scores():
    with open(POST_PATH, encoding='utf-8-sig', newline='') as f:
        return {r['id']: float(r['post_nilai_akhir']) for r in csv.DictReader(f)}

meta  = load_meta()
post  = load_post_scores()

complete_e = [r for r in meta if r['kelompok']=='eksperimen'
              and r['pre_total'] is not None and r['post_total'] is not None]
complete_k = [r for r in meta if r['kelompok']=='kontrol'
              and r['pre_total'] is not None and r['post_total'] is not None]

print(f"Root                : {ROOT}")
print(f"Total data          : {len(meta)}")
print(f"Eksperimen lengkap  : {len(complete_e)}")
print(f"Kontrol lengkap     : {len(complete_k)}")


Root                : /home/primandhika/artikel/dist
Total data          : 77
Eksperimen lengkap  : 40
Kontrol lengkap     : 37


## 1. Validasi Data

In [2]:
issues = []
seen = set()
for row in meta:
    sid = row['id']
    if sid in seen:
        issues.append(f"Duplikat ID: {sid}")
    seen.add(sid)
    for phase in ('pre', 'post'):
        dims = [row.get(f'{phase}_{d}') for d in DIMS]
        total = row.get(f'{phase}_total')
        if any(v is not None for v in dims):
            for d, v in zip(DIMS, dims):
                if v is not None and not (4 <= v <= 16):
                    issues.append(f"{sid} {phase}_{d}={v} di luar [4,16]")
            if total is not None and all(v is not None for v in dims):
                if abs(sum(dims) - total) > 0.01:
                    issues.append(f"{sid} {phase}_total={total} != sum={sum(dims)}")

if issues:
    print("Isu ditemukan:")
    for i in issues: print(" -", i)
else:
    print(f"PASS: {len(meta)} baris valid — semua dimensi dalam [4,16], total konsisten")


PASS: 77 baris valid — semua dimensi dalam [4,16], total konsisten


## 2. Statistik Deskriptif Pre dan Post

In [3]:
def describe(x):
    if not x: return {}
    return {'n':len(x),'mean':mean(x),'sd':stdev(x) if len(x)>1 else 0,
            'median':median(x),'min':min(x),'max':max(x)}

def fmt(v, d=3):
    return f"{v:.{d}f}" if isinstance(v, float) else str(v)

def print_table(headers, rows):
    text = [[fmt(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i,h in enumerate(headers)]
    sep = ' | '
    print(sep.join(str(h).ljust(widths[i]) for i,h in enumerate(headers)))
    print('-+-'.join('-'*w for w in widths))
    for row in text:
        print(sep.join(row[i].ljust(widths[i]) for i in range(len(headers))))

print("=== SKOR TOTAL (maks 64) ===")
rows_d = []
for group, subset in [('Eksperimen', complete_e), ('Kontrol', complete_k)]:
    for phase in ('pre', 'post'):
        vals = [r[f'{phase}_total'] for r in subset if r.get(f'{phase}_total') is not None]
        d = describe(vals)
        rows_d.append([group, phase.capitalize(), d['n'], d['mean'], d['sd'],
                       d['median'], d['min'], d['max']])
print_table(['Kelompok','Fase','n','Mean','SD','Median','Min','Max'], rows_d)

print()
print("=== PER DIMENSI (maks 16) ===")
dim_rows = []
for group, subset in [('Eksperimen', complete_e), ('Kontrol', complete_k)]:
    for dim in DIMS:
        for phase in ('pre', 'post'):
            col = f'{phase}_{dim}'
            vals = [r[col] for r in subset if r.get(col) is not None]
            d = describe(vals)
            dim_rows.append([group, DIM_LABEL[dim], phase.capitalize(),
                             d['n'], d['mean'], d['sd'], d['min'], d['max']])
print_table(['Kelompok','Dimensi','Fase','n','Mean','SD','Min','Max'], dim_rows)


=== SKOR TOTAL (maks 64) ===
Kelompok   | Fase | n  | Mean   | SD     | Median | Min    | Max   
-----------+------+----+--------+--------+--------+--------+-------
Eksperimen | Pre  | 40 | 26.400 | 8.298  | 25.500 | 16.000 | 64.000
Eksperimen | Post | 40 | 50.950 | 9.432  | 52.000 | 16.000 | 64.000
Kontrol    | Pre  | 37 | 31.892 | 11.993 | 27.000 | 20.000 | 62.000
Kontrol    | Post | 37 | 48.351 | 7.454  | 49.000 | 27.000 | 59.000

=== PER DIMENSI (maks 16) ===
Kelompok   | Dimensi              | Fase | n  | Mean   | SD    | Min   | Max   
-----------+----------------------+------+----+--------+-------+-------+-------
Eksperimen | Perencanaan          | Pre  | 40 | 6.275  | 2.670 | 4.000 | 16.000
Eksperimen | Perencanaan          | Post | 40 | 13.100 | 2.827 | 4.000 | 16.000
Eksperimen | Pemantauan           | Pre  | 40 | 6.700  | 2.963 | 4.000 | 16.000
Eksperimen | Pemantauan           | Post | 40 | 12.500 | 2.582 | 4.000 | 16.000
Eksperimen | Evaluasi             | Pre  | 40 | 6.37

## 3. Kategorisasi Skor Metakognitif

Berdasarkan rentang skor total (16–64):

| Kategori | Rentang |
|---|---|
| Sangat Tinggi | 55–64 |
| Tinggi | 43–54 |
| Sedang | 31–42 |
| Rendah | 16–30 |


In [4]:
def kategori(v):
    if v is None: return '-'
    if v >= 55: return 'Sangat Tinggi'
    if v >= 43: return 'Tinggi'
    if v >= 31: return 'Sedang'
    return 'Rendah'

ORDER = ['Rendah','Sedang','Tinggi','Sangat Tinggi']
print("Distribusi kategori skor total:")
for group, subset in [('Eksperimen', complete_e), ('Kontrol', complete_k)]:
    print(f"\n  {group}:")
    for phase in ('pre', 'post'):
        col = f'{phase}_total'
        cats = Counter(kategori(r[col]) for r in subset)
        n = len(subset)
        parts = '  '.join(f"{k}:{cats.get(k,0):>2} ({cats.get(k,0)/n*100:4.1f}%)" for k in ORDER)
        print(f"    {phase.upper()}: {parts}")


Distribusi kategori skor total:

  Eksperimen:
    PRE: Rendah:31 (77.5%)  Sedang: 8 (20.0%)  Tinggi: 0 ( 0.0%)  Sangat Tinggi: 1 ( 2.5%)
    POST: Rendah: 2 ( 5.0%)  Sedang: 1 ( 2.5%)  Tinggi:24 (60.0%)  Sangat Tinggi:13 (32.5%)

  Kontrol:
    PRE: Rendah:25 (67.6%)  Sedang: 5 (13.5%)  Tinggi: 3 ( 8.1%)  Sangat Tinggi: 4 (10.8%)
    POST: Rendah: 1 ( 2.7%)  Sedang: 6 (16.2%)  Tinggi:21 (56.8%)  Sangat Tinggi: 9 (24.3%)


## 4. Gain Absolut dan N-Gain Ternormalisasi (Hake, 1998)

**Gain absolut** = post − pre  
**N-Gain** = (post − pre) / (skor_maks − pre)

Kategori N-Gain (Hake):
- Tinggi : g ≥ 0,70
- Sedang : 0,30 ≤ g < 0,70
- Rendah : g < 0,30


In [5]:
def ngain(pre, post, smax=MAX_TOTAL):
    if pre is None or post is None: return None
    if smax - pre == 0: return None
    return (post - pre) / (smax - pre)

def ngain_kat(g):
    if g is None: return '-'
    if g >= 0.7: return 'Tinggi'
    if g >= 0.3: return 'Sedang'
    return 'Rendah'

gain_e  = [r['post_total'] - r['pre_total'] for r in complete_e]
gain_k  = [r['post_total'] - r['pre_total'] for r in complete_k]
ng_e    = [v for v in (ngain(r['pre_total'], r['post_total']) for r in complete_e) if v is not None]
ng_k    = [v for v in (ngain(r['pre_total'], r['post_total']) for r in complete_k) if v is not None]

print("=== GAIN ABSOLUT ===")
rows_g = []
for label, vals in [('Eksperimen', gain_e), ('Kontrol', gain_k)]:
    d = describe(vals)
    rows_g.append([label, d['n'], d['mean'], d['sd'], d['median'], d['min'], d['max']])
print_table(['Kelompok','n','Mean Gain','SD','Median','Min','Max'], rows_g)

print()
print("=== N-GAIN TERNORMALISASI ===")
rows_ng = []
for label, vals in [('Eksperimen', ng_e), ('Kontrol', ng_k)]:
    d = describe(vals)
    rows_ng.append([label, d['n'], d['mean'], d['sd'], d['median'], d['min'], d['max']])
print_table(['Kelompok','n','Mean N-Gain','SD','Median','Min','Max'], rows_ng)

print()
print("Distribusi kategori N-Gain:")
for group, vals in [('Eksperimen', ng_e), ('Kontrol', ng_k)]:
    cats = Counter(ngain_kat(v) for v in vals)
    n = len(vals)
    parts = '  '.join(f"{k}:{cats.get(k,0)} ({cats.get(k,0)/n*100:.0f}%)"
                     for k in ['Rendah','Sedang','Tinggi'])
    print(f"  {group}: {parts}")


=== GAIN ABSOLUT ===
Kelompok   | n  | Mean Gain | SD     | Median | Min     | Max   
-----------+----+-----------+--------+--------+---------+-------
Eksperimen | 40 | 24.550    | 11.531 | 25.000 | -12.000 | 48.000
Kontrol    | 37 | 16.459    | 10.186 | 18.000 | -4.000  | 30.000

=== N-GAIN TERNORMALISASI ===
Kelompok   | n  | Mean N-Gain | SD    | Median | Min    | Max  
-----------+----+-------------+-------+--------+--------+------
Eksperimen | 39 | 0.648       | 0.249 | 0.676  | -0.333 | 1.000
Kontrol    | 37 | 0.400       | 0.473 | 0.525  | -2.000 | 0.769

Distribusi kategori N-Gain:
  Eksperimen: Rendah:3 (8%)  Sedang:20 (51%)  Tinggi:16 (41%)
  Kontrol: Rendah:10 (27%)  Sedang:20 (54%)  Tinggi:7 (19%)


## 5. Uji Normalitas (Jarque–Bera) pada Gain

Digunakan sebagai diagnostik sebelum memilih uji parametrik vs. nonparametrik.
p_JB besar → distribusi mendekati normal.


In [6]:
def jb_test(x):
    n=len(x); m=mean(x); s=math.sqrt(sum((v-m)**2 for v in x)/n)
    if s == 0: return 0,0,0,1.0
    skew=sum((v-m)**3 for v in x)/n/s**3
    excess=sum((v-m)**4 for v in x)/n/s**4-3
    jb=n/6*(skew**2+excess**2/4)
    return skew,excess,jb,math.exp(-jb/2)

print("Diagnostik normalitas pada distribusi Gain dan N-Gain:")
print()
rows_jb = []
for label, vals in [('Eksperimen Gain',gain_e),('Kontrol Gain',gain_k),
                    ('Eksperimen N-Gain',ng_e),('Kontrol N-Gain',ng_k)]:
    sk,ku,jb,p = jb_test(vals)
    normal = "Mendekati normal" if p > 0.05 else "Non-normal"
    rows_jb.append([label, len(vals), sk, ku, jb, p, normal])
print_table(['Kelompok','n','Skewness','ExcessKurt','JB','p_JB','Keterangan'], rows_jb)


Diagnostik normalitas pada distribusi Gain dan N-Gain:

Kelompok          | n  | Skewness | ExcessKurt | JB      | p_JB  | Keterangan      
------------------+----+----------+------------+---------+-------+-----------------
Eksperimen Gain   | 40 | -0.952   | 1.602      | 10.315  | 0.006 | Non-normal      
Kontrol Gain      | 37 | -0.521   | -0.918     | 2.975   | 0.226 | Mendekati normal
Eksperimen N-Gain | 39 | -1.627   | 4.604      | 51.652  | 0.000 | Non-normal      
Kontrol N-Gain    | 37 | -3.677   | 16.047     | 480.357 | 0.000 | Non-normal      


## 6. Paired Sample t-Test (Perubahan Dalam Kelompok)

Uji t berpasangan — apakah terdapat peningkatan signifikan pre → post dalam masing-masing kelompok.


In [7]:
def betacf(a,b,x):
    qab,qap,qam=a+b,a+1,a-1; c,d=1.0,max(abs(1-qab*x/qap),3e-300)
    d=1/d; h=d
    for m in range(1,201):
        m2=2*m; aa=m*(b-m)*x/((qam+m2)*(a+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        h*=d*c; aa=-(a+m)*(qab+m)*x/((a+m2)*(qap+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        delta=d*c; h*=delta
        if abs(delta-1)<3e-14: break
    return h

def ibeta(a,b,x):
    if x<=0: return 0.0
    if x>=1: return 1.0
    bt=math.exp(math.lgamma(a+b)-math.lgamma(a)-math.lgamma(b)+a*math.log(x)+b*math.log1p(-x))
    return bt*betacf(a,b,x)/a if x<(a+1)/(a+b+2) else 1-bt*betacf(b,a,1-x)/b

def t_p(t,df): return ibeta(df/2,0.5,df/(df+t*t))

def t_crit(df,alpha=.05):
    lo,hi=0.0,20.0
    for _ in range(80):
        mid=(lo+hi)/2
        if t_p(mid,df)>alpha: lo=mid
        else: hi=mid
    return (lo+hi)/2

def paired_t(pre_x, post_x):
    change=[b-a for a,b in zip(pre_x,post_x)]
    n=len(change); se=stdev(change)/math.sqrt(n); t=mean(change)/se; df=n-1; crit=t_crit(df)
    return {'n':n,'mean_gain':mean(change),'ci_low':mean(change)-crit*se,
            'ci_high':mean(change)+crit*se,'t':t,'df':df,'p':t_p(t,df),
            'cohen_dz':mean(change)/stdev(change)}

rows_pt = []
for group, subset in [('Eksperimen', complete_e), ('Kontrol', complete_k)]:
    pre_v  = [r['pre_total'] for r in subset]
    post_v = [r['post_total'] for r in subset]
    r = paired_t(pre_v, post_v)
    sig = "Signifikan" if r['p'] < 0.05 else "Tidak Signifikan"
    rows_pt.append([group, r['n'], r['mean_gain'], r['ci_low'], r['ci_high'],
                    r['t'], r['df'], r['p'], r['cohen_dz'], sig])
print_table(['Kelompok','n','Mean Gain','CI low','CI high','t','df','p','Cohen dz','Keterangan'],
            rows_pt)


Kelompok   | n  | Mean Gain | CI low | CI high | t      | df | p     | Cohen dz | Keterangan
-----------+----+-----------+--------+---------+--------+----+-------+----------+-----------
Eksperimen | 40 | 24.550    | 20.862 | 28.238  | 13.465 | 39 | 0.000 | 2.129    | Signifikan
Kontrol    | 37 | 16.459    | 13.063 | 19.856  | 9.829  | 36 | 0.000 | 1.616    | Signifikan


## 7. Independent Sample t-Test / Welch — Perbedaan N-Gain Antarkelompok

Uji ini menjawab apakah peningkatan metakognitif (N-Gain) kelas eksperimen secara signifikan lebih besar daripada kelas kontrol.


In [8]:
def welch(a, b):
    va,vb,na,nb=variance(a),variance(b),len(a),len(b)
    se=math.sqrt(va/na+vb/nb); diff=mean(a)-mean(b); t=diff/se
    df=(va/na+vb/nb)**2/((va/na)**2/(na-1)+(vb/nb)**2/(nb-1))
    crit=t_crit(df)
    sp=math.sqrt(((na-1)*va+(nb-1)*vb)/(na+nb-2))
    g=diff/sp*(1-3/(4*(na+nb)-9))
    return {'diff':diff,'ci_low':diff-crit*se,'ci_high':diff+crit*se,
            't':t,'df':df,'p':t_p(t,df),'hedges_g':g}

r = welch(ng_e, ng_k)
print("Welch t-test: N-Gain Eksperimen vs Kontrol")
print(f"  n eksperimen        : {len(ng_e)}")
print(f"  n kontrol           : {len(ng_k)}")
print(f"  Mean N-Gain Eks     : {mean(ng_e):.4f}")
print(f"  Mean N-Gain Kont    : {mean(ng_k):.4f}")
print(f"  Selisih             : {r['diff']:.4f}  (95% CI [{r['ci_low']:.4f}, {r['ci_high']:.4f}])")
print(f"  t({r['df']:.2f})           = {r['t']:.4f}")
print(f"  p                   = {r['p']:.5f}")
print(f"  Hedges' g (effect)  = {r['hedges_g']:.4f}")
print()
if r['p'] < 0.05:
    g_label = 'Besar' if abs(r['hedges_g'])>=0.8 else ('Sedang' if abs(r['hedges_g'])>=0.5 else 'Kecil')
    print(f"Kesimpulan: Perbedaan signifikan (p<0.05), effect size {g_label} (g={r['hedges_g']:.3f})")
else:
    print("Kesimpulan: Perbedaan tidak signifikan (p>=0.05)")


Welch t-test: N-Gain Eksperimen vs Kontrol
  n eksperimen        : 39
  n kontrol           : 37
  Mean N-Gain Eks     : 0.6482
  Mean N-Gain Kont    : 0.4005
  Selisih             : 0.2477  (95% CI [0.0724, 0.4231])
  t(53.91)           = 2.8320
  p                   = 0.00649
  Hedges' g (effect)  = 0.6531

Kesimpulan: Perbedaan signifikan (p<0.05), effect size Sedang (g=0.653)


## 8. Analisis Per Dimensi (N-Gain + Welch t-test)

In [9]:
print("Analisis per dimensi (N-Gain Hake, Welch t-test):")
print()
dim_rows2 = []
for dim in DIMS:
    label = DIM_LABEL[dim]
    pre_col, post_col = f'pre_{dim}', f'post_{dim}'
    ng_ed = [ngain(r[pre_col], r[post_col], MAX_DIM) for r in complete_e
             if r[pre_col] is not None and r[post_col] is not None]
    ng_kd = [ngain(r[pre_col], r[post_col], MAX_DIM) for r in complete_k
             if r[pre_col] is not None and r[post_col] is not None]
    ng_ed = [v for v in ng_ed if v is not None]
    ng_kd = [v for v in ng_kd if v is not None]
    if len(ng_ed) < 2 or len(ng_kd) < 2:
        print(f"  {label}: data tidak cukup")
        continue
    r = welch(ng_ed, ng_kd)
    sig = "*" if r['p'] < 0.05 else ""
    dim_rows2.append([label, f"{mean(ng_ed):.3f}", f"{mean(ng_kd):.3f}",
                      f"{r['diff']:.3f}", f"t({r['df']:.1f})={r['t']:.3f}",
                      f"{r['p']:.4f}{sig}", f"{r['hedges_g']:.3f}"])

print_table(['Dimensi','NG_Eks','NG_Kont','Selisih','t(df)','p','Hedges g'], dim_rows2)
print("* = signifikan p<0.05")


Analisis per dimensi (N-Gain Hake, Welch t-test):

Dimensi              | NG_Eks | NG_Kont | Selisih | t(df)         | p       | Hedges g
---------------------+--------+---------+---------+---------------+---------+---------
Perencanaan          | 0.689  | 0.461   | 0.228   | t(51.6)=2.089 | 0.0417* | 0.502   
Pemantauan           | 0.599  | 0.008   | 0.591   | t(33.6)=2.021 | 0.0513  | 0.512   
Evaluasi             | 0.636  | 0.492   | 0.143   | t(64.8)=1.490 | 0.1411  | 0.352   
Integratif (Feynman) | 0.591  | 0.475   | 0.116   | t(68.6)=1.399 | 0.1664  | 0.322   
* = signifikan p<0.05


## 9. Korelasi Metakognitif Post × Postes Berbicara

In [10]:
def pearson_r(x, y):
    n=len(x); mx,my=mean(x),mean(y)
    num=sum((a-mx)*(b-my) for a,b in zip(x,y))
    den=math.sqrt(sum((a-mx)**2 for a in x)*sum((b-my)**2 for b in y))
    return num/den if den else 0

def p_from_r(r, n):
    if abs(r) >= 1: return 0.0
    t = r * math.sqrt(n-2) / math.sqrt(1-r**2)
    return ibeta((n-2)/2, 0.5, (n-2)/(n-2+t**2))

pairs_e = [(r['post_total'], post[r['id']])
           for r in complete_e if r['id'] in post and r['post_total'] is not None]
if pairs_e:
    xm, yp = zip(*pairs_e)
    r_val = pearson_r(list(xm), list(yp))
    p_val = p_from_r(r_val, len(pairs_e))
    interp = 'Kuat' if abs(r_val)>=0.6 else ('Sedang' if abs(r_val)>=0.4 else 'Lemah')
    print(f"Korelasi post_meta x postes berbicara (eksperimen):")
    print(f"  n={len(pairs_e)},  r={r_val:.3f},  p={p_val:.4f}  ({interp})")

# Juga untuk kontrol
pairs_k = [(r['post_total'], post[r['id']])
           for r in complete_k if r['id'] in post and r['post_total'] is not None]
if pairs_k:
    xm, yp = zip(*pairs_k)
    r_val = pearson_r(list(xm), list(yp))
    p_val = p_from_r(r_val, len(pairs_k))
    interp = 'Kuat' if abs(r_val)>=0.6 else ('Sedang' if abs(r_val)>=0.4 else 'Lemah')
    print(f"Korelasi post_meta x postes berbicara (kontrol):")
    print(f"  n={len(pairs_k)},  r={r_val:.3f},  p={p_val:.4f}  ({interp})")


Korelasi post_meta x postes berbicara (eksperimen):
  n=40,  r=-0.007,  p=0.9669  (Lemah)
Korelasi post_meta x postes berbicara (kontrol):
  n=37,  r=0.342,  p=0.0382  (Lemah)


## Catatan Metodologis dan Keterbatasan

1. **Instrumen metakognitif** terdiri atas 16 item (4 dimensi × 4 item, skala Likert 1–4), total skor 16–64.
2. **N-Gain Hake (1998)**: mahasiswa dengan pre = skor_maks dikecualikan dari perhitungan (pembagi = 0). 
3. **Alpha Cronbach** angket metakognitif dihitung terpisah di notebook instrumen.
4. **Kelompok kontrol**: tidak mendapatkan perlakuan media; beberapa mahasiswa memiliki data pre yang sudah lengkap dari awal pengambilan data.
5. **Effect size**: Cohen's dz untuk paired t-test; Hedges' g untuk uji antarkelompok (koreksi bias sampel kecil).
6. **Taraf signifikansi**: α = 0,05 sesuai Bab III.
7. **Desain kuasi-eksperimen**: estimasi kausal rentan terhadap confounding kelas/dosen.



---

## ✍️ Generate Catatan Pembahasan Bab IV

Sel berikut menulis file `catatan_pembahasan_*.md` yang dapat langsung digunakan sebagai bahan draf pembahasan Bab IV.


In [11]:
# ── GENERATE: catatan pembahasan Bab IV untuk metakognitif ─────────────────
import csv, math
from pathlib import Path
from collections import Counter
from statistics import mean, stdev, variance

ROOT = Path.cwd()
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / 'data' / 'field_test' / 'metakognitif.csv').exists():
        ROOT = p; break

META_PATH = ROOT / 'data' / 'field_test' / 'metakognitif.csv'
POST_PATH = ROOT / 'data' / 'field_test' / 'keterampilan_berbicara_postes.csv'
WAW_PATH  = ROOT / 'data' / 'kualitatif' / 'wawancara.csv'

DIMS = ['planning','monitoring','evaluation','integratif']
DIM_LABEL = {'planning':'Perencanaan','monitoring':'Pemantauan',
             'evaluation':'Evaluasi','integratif':'Integratif (Feynman)'}
MAX_TOTAL = 64.0; MAX_DIM = 16.0

def load_meta():
    with open(META_PATH, encoding='utf-8-sig', newline='') as f:
        raw = list(csv.DictReader(f))
    result = []
    for row in raw:
        rec = {'id':row['id'],'kelompok':row['kelompok'],'nama':row['nama']}
        for phase in ('pre','post'):
            for dim in DIMS:
                col = f'{phase}_{dim}'
                rec[col] = float(row[col]) if row.get(col) else None
            rec[f'{phase}_total'] = float(row[f'{phase}_total']) if row.get(f'{phase}_total') else None
        result.append(rec)
    return result

def load_post():
    with open(POST_PATH, encoding='utf-8-sig', newline='') as f:
        return {r['id']: float(r['post_nilai_akhir']) for r in csv.DictReader(f)}

def load_waw():
    with open(WAW_PATH, encoding='utf-8-sig', newline='') as f:
        return [r for r in csv.DictReader(f) if r.get('kode_subjek','').startswith('E')
                and r.get('kutipan')]

def describe(x):
    if not x: return {}
    return dict(n=len(x),mean=mean(x),sd=stdev(x) if len(x)>1 else 0,
                med=sorted(x)[len(x)//2],mn=min(x),mx=max(x))

def ngain(pre,post,smax=MAX_TOTAL):
    if pre is None or post is None or smax-pre==0: return None
    return (post-pre)/(smax-pre)

def ngain_kat(g):
    if g is None: return '-'
    if g>=0.7: return 'Tinggi'
    if g>=0.3: return 'Sedang'
    return 'Rendah'

def betacf(a,b,x):
    qab,qap,qam=a+b,a+1,a-1; c,d=1.0,max(abs(1-qab*x/qap),3e-300)
    d=1/d; h=d
    for m in range(1,201):
        m2=2*m; aa=m*(b-m)*x/((qam+m2)*(a+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        h*=d*c; aa=-(a+m)*(qab+m)*x/((a+m2)*(qap+m2))
        d=1+aa*d; d=1/max(abs(d),3e-300)*(1 if d>=0 else -1)
        c=1+aa/c; c=max(abs(c),3e-300)*(1 if c>=0 else -1)
        delta=d*c; h*=delta
        if abs(delta-1)<3e-14: break
    return h

def ibeta(a,b,x):
    if x<=0: return 0.0
    if x>=1: return 1.0
    bt=math.exp(math.lgamma(a+b)-math.lgamma(a)-math.lgamma(b)+a*math.log(x)+b*math.log1p(-x))
    return bt*betacf(a,b,x)/a if x<(a+1)/(a+b+2) else 1-bt*betacf(b,a,1-x)/b

def t_p(t,df): return ibeta(df/2,0.5,df/(df+t*t))

def t_crit(df,alpha=.05):
    lo,hi=0.0,20.0
    for _ in range(80):
        mid=(lo+hi)/2
        if t_p(mid,df)>alpha: lo=mid
        else: hi=mid
    return (lo+hi)/2

def paired_t(pre_x,post_x):
    change=[b-a for a,b in zip(pre_x,post_x)]
    n=len(change); se=stdev(change)/math.sqrt(n); t=mean(change)/se; df=n-1; crit=t_crit(df)
    return dict(n=n,mean_gain=mean(change),ci_low=mean(change)-crit*se,
                ci_high=mean(change)+crit*se,t=t,df=df,p=t_p(t,df),
                cohen_dz=mean(change)/stdev(change))

def welch(a,b):
    va,vb,na,nb=variance(a),variance(b),len(a),len(b)
    se=math.sqrt(va/na+vb/nb); diff=mean(a)-mean(b); t=diff/se
    df=(va/na+vb/nb)**2/((va/na)**2/(na-1)+(vb/nb)**2/(nb-1))
    crit=t_crit(df)
    sp=math.sqrt(((na-1)*va+(nb-1)*vb)/(na+nb-2))
    g=diff/sp*(1-3/(4*(na+nb)-9))
    return dict(diff=diff,ci_low=diff-crit*se,ci_high=diff+crit*se,
                t=t,df=df,p=t_p(t,df),hedges_g=g)

def pearson_r(x,y):
    n=len(x); mx,my=mean(x),mean(y)
    num=sum((a-mx)*(b-my) for a,b in zip(x,y))
    den=math.sqrt(sum((a-mx)**2 for a in x)*sum((b-my)**2 for b in y))
    return num/den if den else 0

def p_from_r(r,n):
    if abs(r)>=1: return 0.0
    t=r*math.sqrt(n-2)/math.sqrt(1-r**2)
    return ibeta((n-2)/2,0.5,(n-2)/(n-2+t**2))

meta  = load_meta()
post  = load_post()
waw   = load_waw()

comp_e = [r for r in meta if r['kelompok']=='eksperimen'
          and r['pre_total'] is not None and r['post_total'] is not None]
comp_k = [r for r in meta if r['kelompok']=='kontrol'
          and r['pre_total'] is not None and r['post_total'] is not None]

# stats
def kategori(v):
    if v is None: return '-'
    if v>=55: return 'Sangat Tinggi'
    if v>=43: return 'Tinggi'
    if v>=31: return 'Sedang'
    return 'Rendah'

gain_e = [r['post_total']-r['pre_total'] for r in comp_e]
gain_k = [r['post_total']-r['pre_total'] for r in comp_k]
ng_e   = [v for v in (ngain(r['pre_total'],r['post_total']) for r in comp_e) if v is not None]
ng_k   = [v for v in (ngain(r['pre_total'],r['post_total']) for r in comp_k) if v is not None]

de  = describe([r['pre_total']  for r in comp_e])
de2 = describe([r['post_total'] for r in comp_e])
dk  = describe([r['pre_total']  for r in comp_k])
dk2 = describe([r['post_total'] for r in comp_k])

pt_e = paired_t([r['pre_total'] for r in comp_e],[r['post_total'] for r in comp_e])
pt_k = paired_t([r['pre_total'] for r in comp_k],[r['post_total'] for r in comp_k])
w_ng = welch(ng_e, ng_k)

# korelasi post_meta x postes berbicara
pairs_e = [(r['post_total'],post[r['id']]) for r in comp_e if r['id'] in post and r['post_total'] is not None]
pairs_k = [(r['post_total'],post[r['id']]) for r in comp_k if r['id'] in post and r['post_total'] is not None]
r_e = pearson_r([x for x,y in pairs_e],[y for x,y in pairs_e]) if pairs_e else float('nan')
r_k = pearson_r([x for x,y in pairs_k],[y for x,y in pairs_k]) if pairs_k else float('nan')
p_e = p_from_r(r_e,len(pairs_e)) if pairs_e else float('nan')

cat_pre_e  = Counter(kategori(r['pre_total'])  for r in comp_e)
cat_post_e = Counter(kategori(r['post_total']) for r in comp_e)
cat_pre_k  = Counter(kategori(r['pre_total'])  for r in comp_k)
cat_post_k = Counter(kategori(r['post_total']) for r in comp_k)

ng_kat_e = Counter(ngain_kat(v) for v in ng_e)
ng_kat_k = Counter(ngain_kat(v) for v in ng_k)

OUT_PATH = ROOT / 'data' / 'field_test' / 'catatan_pembahasan_metakognitif.md'

sig_e = 'terdapat' if pt_e['p'] < 0.05 else 'tidak terdapat'
sig_ng = 'terdapat' if w_ng['p'] < 0.05 else 'tidak terdapat'
g_label = 'besar' if abs(w_ng['hedges_g'])>=0.8 else ('sedang' if abs(w_ng['hedges_g'])>=0.5 else 'kecil')

ORDER_KAT = ['Rendah','Sedang','Tinggi','Sangat Tinggi']
ORDER_NG  = ['Rendah','Sedang','Tinggi']

lines = [
"# Catatan Pembahasan Bab IV — Kemampuan Metakognitif\n\n",
"> Catatan ini digenerate otomatis dari notebook `olahdata_metakognitif.ipynb`.",
" Gunakan sebagai bahan draf Bab IV bagian E.3 (Pengujian Metakognitif),",
" F (Temuan Kualitatif Mahasiswa), dan G (Integrasi).\n\n",
"---\n\n",

"## A. Deskripsi Data dan Kelengkapan\n\n",
f"Data kemampuan metakognitif pada uji lapangan diperoleh melalui angket pre-postes ",
f"dengan 16 butir pernyataan (4 dimensi × 4 butir, skala 1–4, total skor 16–64). ",
f"Dari 40 mahasiswa kelompok eksperimen dan 37 mahasiswa kelompok kontrol, ",
f"terdapat **{len(comp_e)} pasangan pre-postes lengkap** pada kelompok eksperimen ",
f"dan **{len(comp_k)} pasangan** pada kelompok kontrol.\n\n",

"---\n\n",
"## B. Statistik Deskriptif Pre–Postes\n\n",
"### Tabel: Deskriptif Skor Total Metakognitif (maks 64)\n\n",
"| Kelompok | Fase | n | Mean | SD | Median | Min | Maks |\n",
"|---|---|---:|---:|---:|---:|---:|---:|\n",
f"| Eksperimen | Pre  | {de['n']} | {de['mean']:.2f} | {de['sd']:.2f} | {de['med']:.1f} | {de['mn']:.1f} | {de['mx']:.1f} |\n",
f"| Eksperimen | Post | {de2['n']} | {de2['mean']:.2f} | {de2['sd']:.2f} | {de2['med']:.1f} | {de2['mn']:.1f} | {de2['mx']:.1f} |\n",
f"| Kontrol | Pre  | {dk['n']} | {dk['mean']:.2f} | {dk['sd']:.2f} | {dk['med']:.1f} | {dk['mn']:.1f} | {dk['mx']:.1f} |\n",
f"| Kontrol | Post | {dk2['n']} | {dk2['mean']:.2f} | {dk2['sd']:.2f} | {dk2['med']:.1f} | {dk2['mn']:.1f} | {dk2['mx']:.1f} |\n\n",

f"Rerata skor postes kelompok eksperimen sebesar **{de2['mean']:.2f}** (SD = {de2['sd']:.2f}) ",
f"dan kelompok kontrol sebesar **{dk2['mean']:.2f}** (SD = {dk2['sd']:.2f}). ",
f"Kedua kelompok menunjukkan peningkatan dari prates ke postes, ",
f"dengan kelompok eksperimen memperlihatkan gain rata-rata yang lebih besar ",
f"({mean(gain_e):.2f} vs {mean(gain_k):.2f} poin).\n\n",

"### Tabel: Distribusi Kategori Skor Pre–Post\n\n",
"| Kelompok | Fase | Rendah | Sedang | Tinggi | Sangat Tinggi |\n",
"|---|---|---:|---:|---:|---:|\n",
]
for group, pre_c, post_c, n_tot in [
    ('Eksperimen', cat_pre_e, cat_post_e, len(comp_e)),
    ('Kontrol',    cat_pre_k, cat_post_k, len(comp_k)),
]:
    for phase, cat_d in [('Pre', pre_c), ('Post', post_c)]:
        row_vals = [f"{cat_d.get(k,0)} ({cat_d.get(k,0)/n_tot*100:.0f}%)" for k in ORDER_KAT]
        lines.append(f"| {group} | {phase} | " + " | ".join(row_vals) + " |\n")
lines.append("\n")

lines += [
"---\n\n",
"## C. Gain Absolut dan N-Gain Ternormalisasi (Hake, 1998)\n\n",
"### Tabel: Gain dan N-Gain\n\n",
"| Kelompok | n | Mean Gain | SD Gain | Mean N-Gain | SD N-Gain | Kategori N-Gain |\n",
"|---|---:|---:|---:|---:|---:|---:|\n",
]
ng_e_d = describe(ng_e); ng_k_d = describe(ng_k)
dge = describe(gain_e); dgk = describe(gain_k)

def dominant_kat(cnt, n):
    return max(ORDER_NG, key=lambda k: cnt.get(k,0))

lines += [
f"| Eksperimen | {dge['n']} | {dge['mean']:.3f} | {dge['sd']:.3f} | {ng_e_d['mean']:.3f} | {ng_e_d['sd']:.3f} | {dominant_kat(ng_kat_e,len(ng_e))} |\n",
f"| Kontrol    | {dgk['n']} | {dgk['mean']:.3f} | {dgk['sd']:.3f} | {ng_k_d['mean']:.3f} | {ng_k_d['sd']:.3f} | {dominant_kat(ng_kat_k,len(ng_k))} |\n\n",
]

lines += [
"### Tabel: Distribusi Kategori N-Gain\n\n",
"| Kelompok | Rendah | Sedang | Tinggi |\n",
"|---|---:|---:|---:|\n",
]
for group, ng_kat, ng_list in [('Eksperimen',ng_kat_e,ng_e),('Kontrol',ng_kat_k,ng_k)]:
    n_g = len(ng_list)
    row_vals = [f"{ng_kat.get(k,0)} ({ng_kat.get(k,0)/n_g*100:.0f}%)" for k in ORDER_NG]
    lines.append(f"| {group} | " + " | ".join(row_vals) + " |\n")
lines.append("\n")

lines += [
f"Kelompok eksperimen memperoleh rerata N-Gain sebesar **{ng_e_d['mean']:.3f}** ",
f"yang termasuk kategori **{dominant_kat(ng_kat_e,len(ng_e)).lower()}**, ",
f"sedangkan kelompok kontrol memperoleh N-Gain **{ng_k_d['mean']:.3f}** ",
f"(kategori **{dominant_kat(ng_kat_k,len(ng_k)).lower()}**). ",
"Perbedaan kategori N-Gain ini mengisyaratkan bahwa intervensi media memberi kontribusi ",
"yang lebih besar pada peningkatan kemampuan metakognitif mahasiswa dibandingkan pembelajaran reguler.\n\n",

"---\n\n",
"## D. Uji Beda — Paired Sample t-Test dan Welch t-Test\n\n",
"### Tabel: Hasil Paired Sample t-Test (Pre–Post dalam Kelompok)\n\n",
"| Kelompok | n | Mean Gain | 95% CI | t | df | p | Cohen dz | Keterangan |\n",
"|---|---:|---:|---|---:|---:|---:|---:|---:|\n",
f"| Eksperimen | {pt_e['n']} | {pt_e['mean_gain']:.3f} | [{pt_e['ci_low']:.3f}; {pt_e['ci_high']:.3f}] | {pt_e['t']:.3f} | {pt_e['df']} | {pt_e['p']:.4f} | {pt_e['cohen_dz']:.3f} | {'Signifikan' if pt_e['p']<0.05 else 'Tidak Signifikan'} |\n",
f"| Kontrol | {pt_k['n']} | {pt_k['mean_gain']:.3f} | [{pt_k['ci_low']:.3f}; {pt_k['ci_high']:.3f}] | {pt_k['t']:.3f} | {pt_k['df']} | {pt_k['p']:.4f} | {pt_k['cohen_dz']:.3f} | {'Signifikan' if pt_k['p']<0.05 else 'Tidak Signifikan'} |\n\n",

"### Tabel: Perbandingan N-Gain Antarkelompok (Welch t-Test)\n\n",
"| Selisih N-Gain | 95% CI | t | df | p | Hedges g | Keterangan |\n",
"|---:|---|---:|---:|---:|---:|---:|\n",
f"| {w_ng['diff']:.4f} | [{w_ng['ci_low']:.4f}; {w_ng['ci_high']:.4f}] | {w_ng['t']:.3f} | {w_ng['df']:.2f} | {w_ng['p']:.5f} | {w_ng['hedges_g']:.3f} | {'Signifikan' if w_ng['p']<0.05 else 'Tidak Signifikan'} |\n\n",

f"Hasil paired t-test menunjukkan bahwa **{sig_e} peningkatan yang signifikan** ",
f"pada kelompok eksperimen (t = {pt_e['t']:.3f}, p = {pt_e['p']:.4f}, Cohen dz = {pt_e['cohen_dz']:.3f}). ",
f"Perbandingan N-Gain antarkelompok dengan Welch t-test menunjukkan bahwa ",
f"**{sig_ng} perbedaan yang signifikan** antara kedua kelompok ",
f"(t({w_ng['df']:.2f}) = {w_ng['t']:.3f}, p = {w_ng['p']:.5f}, Hedges g = {w_ng['hedges_g']:.3f} — efek **{g_label}**).\n\n",
"Temuan ini ",
("mendukung bahwa media web *microlearning* berbasis teknik Feynman memberi kontribusi lebih besar "
 "terhadap peningkatan kemampuan metakognitif mahasiswa dibandingkan pembelajaran reguler. "
 "Hasil ini selaras dengan pandangan Paterson (2022) dan Goh & Liu (2023) bahwa strategi pembelajaran "
 "yang menekankan refleksi diri dan pemantauan performa dapat memperkuat dimensi metakognitif mahasiswa.\n\n"
 if w_ng['p']<0.05 else
 "belum sepenuhnya mendukung perbedaan efektif antara kedua kelompok secara statistik. "
 "Interpretasi ini harus dibatasi dan dilengkapi dengan temuan kualitatif yang dapat menjelaskan "
 "pola perubahan yang belum tertangkap secara numerik.\n\n"),

"---\n\n",
"## E. Korelasi Metakognitif Postes × Keterampilan Berbicara Postes\n\n",
"| Kelompok | n | r Pearson | p | Interpretasi |\n",
"|---|---:|---:|---:|---:|\n",
]
if pairs_e:
    r_interp_e = 'Kuat' if abs(r_e)>=0.6 else ('Sedang' if abs(r_e)>=0.4 else 'Lemah')
    lines.append(f"| Eksperimen | {len(pairs_e)} | {r_e:.3f} | {p_e:.4f} | {r_interp_e} |\n")
if pairs_k:
    p_k = p_from_r(r_k,len(pairs_k))
    r_interp_k = 'Kuat' if abs(r_k)>=0.6 else ('Sedang' if abs(r_k)>=0.4 else 'Lemah')
    lines.append(f"| Kontrol | {len(pairs_k)} | {r_k:.3f} | {p_k:.4f} | {r_interp_k} |\n")
lines.append("\n")

lines += [
"Analisis korelasi ini bersifat eksploratif. Keterbatasan data pre-postes metakognitif ",
"menjadikan interpretasi korelasi ini belum konklusif. Penjelasan yang lebih mendalam ",
"perlu dilakukan melalui triangulasi dengan data wawancara dan observasi.\n\n",

"---\n\n",
"## F. Temuan Kualitatif — Kutipan Wawancara Mahasiswa\n\n",
f"Wawancara kualitatif dilakukan kepada {len(waw)} mahasiswa terpilih dari kelompok eksperimen. ",
"Kutipan-kutipan berikut diorganisir berdasarkan tema dan subtema.\n\n",
]

tema_map = {}
for row in waw:
    tema = row.get('tema','')
    subtema = row.get('subtema','')
    kutipan = row.get('kutipan','')
    nama = row.get('nama','')
    catatan = row.get('catatan_analisis','')
    if tema not in tema_map:
        tema_map[tema] = []
    tema_map[tema].append((subtema, nama, kutipan, catatan))

for tema, entries in tema_map.items():
    lines.append(f"### {tema}\n\n")
    for subtema, nama, kutipan, catatan in entries:
        lines.append(f"**Subtema: {subtema}** ({nama})\n\n")
        if kutipan:
            lines.append(f'> *"{kutipan}"*\n\n')
        if catatan:
            lines.append(f"_{catatan}_\n\n")

lines += [
"---\n\n",
"## G. Ringkasan Integratif untuk Pembahasan Bab IV\n\n",
"Berdasarkan data kuantitatif (angket pre-postes metakognitif) dan kualitatif (wawancara mahasiswa), ",
"berikut poin-poin yang dapat dikembangkan dalam pembahasan Bab IV:\n\n",
f"1. **Peningkatan dalam kelompok eksperimen** {sig_e} secara statistik (p = {pt_e['p']:.4f}), ",
f"dengan Cohen dz = {pt_e['cohen_dz']:.3f} yang mengindikasikan efek yang ",
f"{'besar' if pt_e['cohen_dz']>=0.8 else ('sedang' if pt_e['cohen_dz']>=0.5 else 'kecil')}.\n\n",
f"2. **N-Gain eksperimen** sebesar {ng_e_d['mean']:.3f} (kategori {dominant_kat(ng_kat_e,len(ng_e))}), ",
f"lebih tinggi dari N-Gain kontrol {ng_k_d['mean']:.3f} (kategori {dominant_kat(ng_kat_k,len(ng_k))}), ",
f"dengan perbedaan {'yang bermakna' if w_ng['p']<0.05 else 'yang belum signifikan secara statistik'} ",
f"(Hedges g = {w_ng['hedges_g']:.3f}).\n\n",
"3. **Dimensi metakognitif yang paling berkembang** dapat diidentifikasi dari analisis per dimensi ",
"di notebook, dan perlu didukung kutipan wawancara mahasiswa yang relevan (lihat Bagian F di atas).\n\n",
"4. **Kutipan wawancara mahasiswa** di atas dapat digunakan sebagai bukti kualitatif yang menjelaskan ",
"mekanisme peningkatan metakognitif: dari perencanaan, pemantauan diri, hingga refleksi iteratif ",
"melalui fitur playback dan umpan balik media.\n\n",
"5. **Keterbatasan data**: tidak semua mahasiswa memiliki data pre lengkap, sehingga estimasi efektivitas ",
"pada konstruk ini perlu dibahas dengan kehati-hatian sesuai prinsip transparansi pelaporan ilmiah.\n\n",
"---\n",
f"*File ini digenerate dari `olahdata_metakognitif.ipynb` — {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M')}*\n",
]

OUT_PATH.write_text("".join(lines), encoding='utf-8')
print(f"OK  {OUT_PATH}")
print(f"    Eks: pre={de['mean']:.2f}->post={de2['mean']:.2f} (gain={mean(gain_e):.2f}, N-Gain={ng_e_d['mean']:.3f})")
print(f"    Kont: pre={dk['mean']:.2f}->post={dk2['mean']:.2f} (gain={mean(gain_k):.2f}, N-Gain={ng_k_d['mean']:.3f})")
print(f"    Welch N-Gain: t={w_ng['t']:.3f}, p={w_ng['p']:.5f}, g={w_ng['hedges_g']:.3f}")
print(f"    Wawancara mahasiswa: {len(waw)} kutipan")


OK  /home/primandhika/artikel/dist/data/field_test/catatan_pembahasan_metakognitif.md
    Eks: pre=26.40->post=50.95 (gain=24.55, N-Gain=0.648)
    Kont: pre=31.89->post=48.35 (gain=16.46, N-Gain=0.400)
    Welch N-Gain: t=2.832, p=0.00649, g=0.653
    Wawancara mahasiswa: 12 kutipan
